In [1]:
import os
os.chdir(r"D:\Projects\Poverty Predictor Bd")
os.makedirs("docs", exist_ok=True)
print("Ready")

Ready


In [2]:
content = """# Data Dictionary

All features used in the Bangladesh Poverty Prediction model.
Generated from `master_features.csv` — 64 districts, 30 columns.

---

## Identity Columns

| Column | Type | Description |
|--------|------|-------------|
| `district_name` | string | District name in BBS standard spelling (64 districts) |
| `division_name` | string | Division the district belongs to (8 divisions) |

---

## Nighttime Light Features
**Source:** VIIRS DNB Monthly Composites (NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG)  
**Extracted via:** Google Earth Engine `reduceRegions()`  
**Period:** 2022 annual median composite  
**Unit:** Radiance (nW/cm²/sr)

| Column | Description |
|--------|-------------|
| `ntl_mean` | Mean nighttime light radiance across the district. Core economic activity proxy. |
| `ntl_std` | Standard deviation of light within the district. High = uneven urban-rural development. |
| `ntl_max` | Maximum radiance pixel. Captures the brightest urban core. |
| `ntl_min` | Minimum radiance. Near zero in rural areas. |
| `ntl_p25` | 25th percentile of radiance. Represents darker/poorer areas within the district. |
| `ntl_p75` | 75th percentile of radiance. Represents brighter/wealthier areas. |
| `ntl_iqr` | Interquartile range (p75 - p25). Measures inequality of light distribution. |
| `ntl_yoy_change` | Year-on-year change in mean NTL from 2021 to 2022. Recent economic momentum. |
| `ntl_trend` | Linear slope of NTL from 2018 to 2022. Positive = growing economy. |

---

## Population Features
**Source:** WorldPop GP 100m Population Density (WorldPop/GP/100m/pop)  
**Extracted via:** Google Earth Engine  
**Period:** 2020  
**Unit:** People per 100m x 100m pixel

| Column | Description |
|--------|-------------|
| `pop_density` | Average population density per pixel across the district. |
| `pop_total` | Total estimated population of the district (sum of all pixels). |

---

## Vegetation Feature
**Source:** MODIS Terra Vegetation Indices (MODIS/061/MOD13A3)  
**Extracted via:** Google Earth Engine  
**Period:** 2022 annual mean  
**Unit:** NDVI (dimensionless, -1 to 1)

| Column | Description |
|--------|-------------|
| `ndvi_mean` | Mean Normalized Difference Vegetation Index. Higher = more vegetation. Rural/agricultural proxy. |
| `ndvi_std` | Variation in vegetation within the district. Mixed urban-rural districts have higher std. |

---

## Land Cover Features
**Source:** ESA WorldCover v200 (ESA/WorldCover/v200)  
**Extracted via:** Google Earth Engine  
**Period:** 2021  
**Unit:** Fraction (0 to 1)

| Column | Description |
|--------|-------------|
| `urban_fraction` | Proportion of district classified as urban land (class 50). Strong poverty correlate. |
| `water_fraction` | Proportion of district covered by water bodies (class 80). High in coastal/delta districts. |

---

## Terrain Features
**Source:** SRTM Digital Elevation Model (USGS/SRTMGL1_003)  
**Extracted via:** Google Earth Engine  
**Unit:** Metres above sea level

| Column | Description |
|--------|-------------|
| `elevation_mean` | Average elevation. Low = coastal/delta. High = Chittagong Hill Tracts. |
| `elevation_std` | Variation in elevation. High = hilly terrain. Low = flat delta plains. |

---

## Infrastructure Features
**Source:** OpenStreetMap via OSMnx Python library  
**Extracted:** March 2024  
**Unit:** Kilometres / km²

| Column | Description |
|--------|-------------|
| `road_length_km` | Total length of driveable roads in the district (km). |
| `road_density` | Road length per sq km of district area. Development proxy normalized for size. |
| `area_sqkm` | District area in square kilometres (from projected geometry). |

---

## Engineered Features

| Column | Description |
|--------|-------------|
| `ntl_per_capita` | Mean NTL divided by population density. Economic activity per person. |
| `ntl_mean_spatial_lag` | Weighted average of neighbouring districts' NTL mean (Queen contiguity weights). |
| `ntl_per_capita_spatial_lag` | Weighted average of neighbouring districts' NTL per capita. |
| `ndvi_mean_spatial_lag` | Weighted average of neighbouring districts' NDVI mean. |
| `pop_density_spatial_lag` | Weighted average of neighbouring districts' population density. |

---

## Target Variables
**Source:** Household Income and Expenditure Survey 2022 (HIES 2022)  
**Publisher:** Bangladesh Bureau of Statistics (BBS)  
**Note:** Labels are at division level (8 divisions). All districts within a division share the same label.

| Column | Description |
|--------|-------------|
| `poverty_hcr` | **Main target Y.** Headcount ratio (%) — population below upper poverty line (2022). |
| `poverty_hcr_lower` | Headcount ratio below the lower (extreme) poverty line. |
| `poverty_change` | Change in upper HCR from 2016 to 2022. Negative = poverty reduced. |

---

## Model Output

| Column | Description |
|--------|-------------|
| `poverty_predicted` | Random Forest model prediction of poverty_hcr (in-sample fit). |
"""

with open("docs/data_dictionary.md", "w", encoding="utf-8") as f:
    f.write(content)
print("Saved: docs/data_dictionary.md")

Saved: docs/data_dictionary.md


In [3]:
content = """# Data Sources

All datasets used in this project. Every source is documented
for reproducibility and academic citation.

---

## Ground Truth / Target Variable

| Field | Detail |
|-------|--------|
| **Dataset** | Household Income and Expenditure Survey (HIES) 2022 |
| **Publisher** | Bangladesh Bureau of Statistics (BBS) |
| **URL** | https://bbs.gov.bd |
| **Access date** | January 2024 |
| **Variable used** | Upper poverty line headcount ratio by division |
| **Spatial resolution** | Division level (8 divisions) |
| **Notes** | Table 6.4, page 98 of the full report PDF |

---

## Satellite Features

### Nighttime Light (NTL)
| Field | Detail |
|-------|--------|
| **Dataset** | VIIRS Day/Night Band Monthly Composites V1 |
| **Collection ID** | NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG |
| **Platform** | Google Earth Engine |
| **URL** | https://developers.google.com/earth-engine/datasets/catalog/NOAA_VIIRS_DNB_MONTHLY_V1_VCMCFG |
| **Temporal coverage** | 2018–2022 (annual median composites) |
| **Spatial resolution** | ~500m |
| **Access date** | February 2024 |

### Vegetation Index (NDVI)
| Field | Detail |
|-------|--------|
| **Dataset** | MODIS Terra Vegetation Indices 1km Monthly |
| **Collection ID** | MODIS/061/MOD13A3 |
| **Platform** | Google Earth Engine |
| **URL** | https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A3 |
| **Temporal coverage** | 2022 annual mean |
| **Spatial resolution** | 1km |
| **Access date** | February 2024 |

### Land Cover
| Field | Detail |
|-------|--------|
| **Dataset** | ESA WorldCover v200 |
| **Collection ID** | ESA/WorldCover/v200 |
| **Platform** | Google Earth Engine |
| **URL** | https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200 |
| **Temporal coverage** | 2021 |
| **Spatial resolution** | 10m |
| **Access date** | February 2024 |

### Elevation (SRTM)
| Field | Detail |
|-------|--------|
| **Dataset** | SRTM Digital Elevation Data Version 4 |
| **Collection ID** | USGS/SRTMGL1_003 |
| **Platform** | Google Earth Engine |
| **URL** | https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003 |
| **Spatial resolution** | 30m |
| **Access date** | February 2024 |

### Sentinel-2 Satellite Imagery (CNN phase)
| Field | Detail |
|-------|--------|
| **Dataset** | Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (Harmonized) |
| **Collection ID** | COPERNICUS/S2_SR_HARMONIZED |
| **Platform** | Google Earth Engine |
| **URL** | https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED |
| **Temporal coverage** | Nov 2022 – Mar 2023 (dry season, cloud-free) |
| **Spatial resolution** | 100m (resampled from 10m) |
| **Bands used** | B4 (Red), B3 (Green), B2 (Blue), B8 (NIR) |
| **Access date** | March 2024 |

---

## Population

| Field | Detail |
|-------|--------|
| **Dataset** | WorldPop Global Population Density 100m |
| **Collection ID** | WorldPop/GP/100m/pop |
| **Platform** | Google Earth Engine |
| **URL** | https://www.worldpop.org |
| **Temporal coverage** | 2020 |
| **Spatial resolution** | 100m |
| **Access date** | February 2024 |

---

## Administrative Boundaries

| Field | Detail |
|-------|--------|
| **Dataset** | GADM Bangladesh Level 2 (Districts) |
| **Version** | GADM 4.1 |
| **URL** | https://gadm.org/download_country.html |
| **Access date** | January 2024 |
| **Notes** | 64 districts. Some names standardized to BBS spelling (see src/name_mapping.py) |

---

## Road Network

| Field | Detail |
|-------|--------|
| **Dataset** | OpenStreetMap road network |
| **Tool** | OSMnx Python library v1.9.3 |
| **URL** | https://www.openstreetmap.org |
| **Access date** | March 2024 |
| **Network type** | Drive (driveable roads only) |

---

## Secondary / Cross-Validation

| Field | Detail |
|-------|--------|
| **Dataset** | World Bank Poverty Map of Bangladesh |
| **URL** | https://data.worldbank.org |
| **Access date** | January 2024 |
| **Notes** | Used for cross-validation of poverty labels |

---

## GEE Project
All Google Earth Engine extractions were performed under 
GEE Cloud Project: `poverty-prediction-489716`
"""

with open("docs/data_sources.md", "w", encoding="utf-8") as f:
    f.write(content)
print("Saved: docs/data_sources.md")

Saved: docs/data_sources.md


In [4]:
content = """# Methodology

Decisions made throughout the project, alternatives considered,
and justifications. This document forms the basis of the
Methods section of the research paper.

---

## 1. Problem Framing

**Decision:** Treat poverty prediction as a regression problem 
predicting the Headcount Ratio (HCR %) rather than a 
classification problem.

**Reason:** HCR is a continuous variable. Regression preserves 
the ordering and magnitude of poverty differences between 
divisions. Classification would lose information.

**Limitation acknowledged:** HIES 2022 only reports poverty at 
division level (8 values). All 64 districts within a division 
share the same label. This is the core challenge — we are 
essentially learning to predict 8 distinct values from 
district-level satellite features.

---

## 2. Spatial Unit of Analysis

**Decision:** Use districts (64) as the spatial unit, not 
divisions (8) or upazilas (490+).

**Reason:** 
- Divisions (8) are too few for ML — any model would 
  simply memorize 8 values.
- Upazilas (490+) have no reliable poverty labels available.
- Districts (64) give enough samples for meaningful ML while 
  having reasonably stable satellite feature aggregations.

---

## 3. Target Variable

**Decision:** Use upper poverty line HCR from HIES 2022.

**Alternatives considered:**
- Lower poverty line HCR — captures extreme poverty but 
  misses the moderately poor.
- Consumption expenditure — more granular but not publicly 
  available at district level.
- World Bank estimates — used as secondary cross-validation 
  source.

---

## 4. Feature Engineering

### NTL Temporal Features
**Decision:** Extract 5 years (2018–2022) to compute trend 
and year-on-year change rather than using single-year NTL.

**Reason:** Economic trajectory matters — a district with 
declining NTL may have higher poverty than its current 
absolute NTL suggests.

### NTL per Capita
**Decision:** Divide NTL by population density to create 
NTL per capita.

**Reason:** Raw NTL correlates with population size, not just 
wealth. Dhaka has high NTL partly because it has 10 million 
people, not just because it is wealthy.

### Spatial Lag Features
**Decision:** Compute Queen contiguity spatial lag for 
ntl_mean, ntl_per_capita, ndvi_mean, pop_density.

**Reason:** Moran's I test showed poverty has strong spatial 
autocorrelation (I = 0.733, p < 0.001). Neighboring districts' 
features are informative about a district's poverty level.

**Impact:** Spatial lag features contributed ~29% of total 
Random Forest feature importance, validating this decision.

---

## 5. Evaluation Strategy

**Decision:** Use Leave-One-Division-Out (LODO) 
cross-validation rather than standard k-fold.

**Reason:** Standard k-fold randomly splits districts, meaning 
training and test sets contain neighboring districts. Since 
poverty and satellite features are spatially autocorrelated 
(Moran's I = 0.733), this leaks information and inflates 
performance estimates.

**LODO approach:** Train on 7 divisions, test on 1, rotate 
all 8 times. This simulates predicting poverty in a genuinely 
unseen region.

**Limitation:** With only 8 folds, variance between folds is 
high. Some divisions (Mymensingh, Sylhet) have only 4 districts 
in the test set, making per-fold R² unstable.

---

## 6. Model Selection

### Baseline Models
Linear regression, Ridge, and Lasso were used as baselines. 
All performed worse than the naive mean predictor under LODO-CV 
(RMSE 16–28pp) due to high variance extrapolation — the linear 
models could not handle the large feature values of Dhaka 
district when trained on other divisions.

### Tree-Based Models
Random Forest (RMSE 3.626) and LightGBM (RMSE 3.673) both 
beat the naive baseline. Random Forest was selected as the 
final model due to:
- Marginally better RMSE
- More stable predictions (lower variance across folds)
- Better interpretability via SHAP

### Hyperparameter Tuning
GridSearchCV with LODO-CV was used. Best parameters:
`max_depth=None, max_features=0.7, min_samples_leaf=1, 
n_estimators=100`

### CNN Extension
ResNet-18 was fine-tuned on 4-band Sentinel-2 images 
(100m resolution, 224x224px). Despite using transfer learning 
from ImageNet, the CNN performed worse than RF under LODO-CV 
(RMSE 4.354 vs 3.626). This is expected with only 64 training 
samples — CNNs require thousands of examples to generalize.

**Key finding:** The CNN achieved better MAE (3.19 vs 2.93), 
suggesting it makes fewer large errors on well-represented 
divisions but struggles more on outlier divisions (Barishal, 
Rangpur).

---

## 7. Feature Importance Findings

Top features by Random Forest importance:
1. elevation_mean (19.2%) — terrain geography
2. ntl_mean_spatial_lag (17.3%) — neighboring NTL context
3. ntl_iqr (9.3%) — within-district light inequality
4. road_density (9.2%) — infrastructure
5. ntl_yoy_change (6.5%) — economic momentum

**Key insight:** The spatial lag of NTL (17.3%) is more 
important than the district's own NTL (2.3%). This means 
where a district is located matters more than its own 
nighttime light intensity.

---

## 8. Error Analysis

The model systematically underpredicts poverty in Barishal 
(true: 26.9%, predicted: ~21–22%) and Rangpur (true: 24.8%, 
predicted: ~20–21%).

**Possible explanations:**
- Barishal is a delta region with high water coverage — the 
  satellite signal may be attenuated by water bodies.
- Rangpur has unusually high road density for its poverty 
  level (rural feeder roads), which confounds the model.
- Both divisions have few districts (6 and 8) — the model 
  has less training data for these poverty levels.

---

## 9. Limitations

1. **Division-level labels:** 8 unique target values across 
   64 districts constrains model learning.
2. **Small sample size:** 64 districts is insufficient for 
   deep learning. CNN results should be interpreted cautiously.
3. **Cross-sectional:** Only 2022 data used for the final 
   model. Temporal poverty dynamics are not fully captured.
4. **Label mismatch:** HIES 2022 measures consumption poverty; 
   satellite data measures economic activity proxies. The 
   relationship is correlational, not causal.
5. **Spatial resolution:** Some small districts 
   (Narayanganj: 332x619 pixels at 100m) have limited 
   satellite coverage for CNN.
"""

with open("docs/methodology.md", "w", encoding="utf-8") as f:
    f.write(content)
print("Saved: docs/methodology.md")

Saved: docs/methodology.md


In [5]:
content = """# 🛰️ Mapping Poverty from Space — Bangladesh

> Predicting district-level poverty in Bangladesh using satellite 
> nighttime light, vegetation indices, terrain data, and road 
> networks — without a single ground survey.

**ISRT · University of Dhaka · 2024**

[![Streamlit App](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](https://povertypredictionbd-phjajjhjhoxmrkzgy7kxod.streamlit.app)

---

## 🗺️ Key Result

![Poverty Prediction Map](outputs/maps/poverty_prediction_map.png)

*Left: Actual poverty rates (HIES 2022). Centre: RF model 
predictions. Right: Prediction error by district.*

---

## 📊 Model Performance

| Model | RMSE (pp) | MAE (pp) | R² | Validation |
|-------|-----------|----------|-----|------------|
| Naive Baseline | 4.163 | 3.596 | 0.000 | LODO-CV |
| CNN ResNet-18 | 4.354 | 3.188 | -0.094 | LODO-CV |
| **Random Forest** | **3.626** | **2.926** | **0.241** | LODO-CV |

*LODO-CV = Leave-One-Division-Out Cross-Validation*

---

## 🔬 Key Findings

- **Poverty is strongly spatially clustered** — Moran's I = 0.733 
  (p < 0.001). Poor districts neighbor poor districts.
- **Neighboring NTL matters more than own NTL** — spatial lag 
  features contribute 29% of total RF importance.
- **Elevation dominates** (19.2% importance) — terrain geography 
  shapes economic development across Bangladesh.
- **CNN underperforms RF** with only 64 training samples — 
  tabular ML outperforms deep learning at this data scale.

---

## 📁 Project Structure
```
bangladesh-poverty-prediction/
├── data/
│   ├── raw/               # Raw GEE exports, shapefiles
│   └── processed/         # Master feature dataset
├── models/                # Trained RF model + scaler
├── notebooks/             # Analysis notebooks (01–10)
├── outputs/
│   ├── figures/           # Charts and visualizations
│   ├── maps/              # Choropleth maps
│   └── tables/            # Results tables
├── src/                   # Reusable Python modules
├── docs/                  # Documentation
├── lovable_app/           # React frontend + FastAPI backend
├── app.py                 # Streamlit dashboard
└── requirements.txt       # Python dependencies
```

---

## 🗂️ Data Sources

| Dataset | Source | Resolution |
|---------|--------|------------|
| Poverty labels | HIES 2022, Bangladesh BBS | Division level |
| Nighttime Light | VIIRS DNB, Google Earth Engine | ~500m |
| Vegetation (NDVI) | MODIS MOD13A3, GEE | 1km |
| Land Cover | ESA WorldCover v200, GEE | 10m |
| Elevation | SRTM, GEE | 30m |
| Population | WorldPop GP, GEE | 100m |
| Road Network | OpenStreetMap, OSMnx | Vector |
| Satellite Imagery | Sentinel-2, GEE | 100m |
| Boundaries | GADM 4.1 Level 2 | Vector |

---

## 🚀 How to Reproduce
```bash
# 1. Clone the repository
git clone https://github.com/raiyan-ahmed-khan/Poverty_Prediction_BD
cd Poverty_Prediction_BD

# 2. Create virtual environment
python -m venv .venv
source .venv/bin/activate  # Windows: .venv\\Scripts\\activate

# 3. Install dependencies
pip install -r requirements.txt

# 4. Run notebooks in order
# notebooks/01 → boundary verification
# notebooks/02 → nighttime light extraction (GEE)
# notebooks/03 → auxiliary features
# notebooks/04 → data merging
# notebooks/05 → EDA
# notebooks/06 → ML modeling
# notebooks/07 → Sentinel-2 export (GEE)
# notebooks/08 → SHAP analysis
# notebooks/09 → Lovable app preparation

# 5. Launch Streamlit dashboard
streamlit run app.py
```

---

## 🌐 Live Demo

- **Streamlit Dashboard:** https://povertypredictionbd-phjajjhjhoxmrkzgy7kxod.streamlit.app
- **API Docs:** https://bangladesh-poverty-api.onrender.com/docs

---

## 📄 Citation
```bibtex
@misc{khan2024poverty,
  author    = {Raiyan Ahmed Khan},
  title     = {Predicting Regional Poverty Levels in Bangladesh 
               Using Satellite Night-Light Data and Geospatial Features},
  year      = {2024},
  publisher = {ISRT, University of Dhaka},
  url       = {https://github.com/raiyan-ahmed-khan/Poverty_Prediction_BD}
}
```

---

## 🙏 Acknowledgements

- **Supervisor:** ISRT, University of Dhaka
- **Data:** Bangladesh Bureau of Statistics, Google Earth Engine, 
  ESA, NASA, OpenStreetMap contributors, WorldPop
- **Inspiration:** Jean et al. (2016) *Combining satellite imagery 
  and machine learning to predict poverty.* Science.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(content)
print("Saved: README.md")

Saved: README.md


In [6]:
import subprocess

result = subprocess.run(
    ['git', 'add', 'docs/', 'README.md', 'notebooks/'],
    capture_output=True, text=True
)
print(result.stdout or "Files staged")

result = subprocess.run(
    ['git', 'commit', '-m', 
     'Complete Phase 6 - data dictionary, methodology, README'],
    capture_output=True, text=True
)
print(result.stdout)

Files staged
[main 6f362f2] Complete Phase 6 - data dictionary, methodology, README
 6 files changed, 628 insertions(+), 1 deletion(-)
 create mode 100644 docs/data_dictionary.md
 create mode 100644 docs/data_sources.md
 create mode 100644 docs/methodology.md
 create mode 100644 notebooks/10_documentation.ipynb

